# UMAP / t-SNE embedding visualisation

## 1 · Config

In [1]:
import sys
sys.path.append("..")

In [2]:
from pathlib import Path
from paths import load_paths

### Model config

In [3]:
MODEL_NAME = "satclip_open_clip_vit_l"
CONCEPT_DATASET = "git-10m"
CONCEPT_NAME = "geospatial_all"

_paths = load_paths()
MODEL_PATH  = str(_paths["models"][MODEL_NAME]["model_path"])
CONCEPT_SET = str(_paths["concept_sets"][CONCEPT_DATASET][CONCEPT_NAME])

### Output Config

In [4]:
OUTPUT_DIR = Path("figures/umap") / MODEL_NAME / CONCEPT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Data config

In [5]:
LOCATIONS_PATH = Path("/home/libe2152/data/dense_grid/dense_grid.csv")
LAT_COL, LON_COL = "lat", "lon"

### Plot config

In [6]:
NUM_SAMPLES = 10_000  # None → full dataset
STRATIFY_BY_CONTINENT = True

## 2 · Imports & model/data loading

In [7]:
import json, sys
import numpy as np, pandas as pd, torch
import matplotlib, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from plot import (
    load_model, load_locations, sample_locations,
    embed_locations, embed_concepts,
    center_renorm, get_continents, umap_reduce
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [8]:
df = load_locations(LOCATIONS_PATH, MODEL_PATH)
df = sample_locations(df, NUM_SAMPLES, STRATIFY_BY_CONTINENT)
print(f"Locations: {len(df):,}")

loc_precomputed = "location_embedding" in df.columns
model, model_args = load_model(MODEL_PATH, device, loc_precomputed=loc_precomputed)
print("Model args:", vars(model_args))

/home/libe2152/projects/explainable-earth-embeddings/eval_results/plot.py:111: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), n_per), random_state=0))
/home/libe2152/miniconda3/envs/fai/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


Locations: 8,762


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_

using pretrained moco vit16
Model args: {'dataset_name': 'git-10M', 'dataset_path': '/home/libe2152/data/git-10M/satclip/', 'precomputed_dir': None, 'train_subsample_size': None, 'val_subsample_size': None, 'precomputed_text_embeddings': False, 'precomputed_location_embeddings': True, 'text_encoder': 'open_clip_vit_l', 'location_encoder': 'satclip', 'text_finetune_mode': 'lora', 'loc_finetune_mode': 'only_proj', 'lora_rank': '8', 'lora_layers': 8, 'text_projection': 'linear', 'text_proj_hidden_layers': 1, 'text_proj_hidden_features': 2048, 'location_projection': 'none', 'loc_proj_hidden_layers': 1, 'loc_proj_hidden_features': 512, 'shared_dim': 256, 'train_loss': 'clip_symmetric', 'lambda_alignment': 1.0, 'sigma': 1.0, 'logit_scale_temp': 0.07, 'max_steps': 100000, 'val_every_n_steps': 500, 'num_val_checks_without_improvement': 5, 'batch_size': 2048, 'lr': 0.0001, 'scheduler': 'cosine', 'warmup_steps': 0, 'weight_decay': 0.01, 'text_nonlinearity': 'gelu', 'loc_nonlinearity': 'sine', 'a

## 3 · Compute embeddings

In [9]:
loc_emb = embed_locations(model, df, device, lat_col=LAT_COL, lon_col=LON_COL)

concepts_raw = json.loads(Path(CONCEPT_SET).read_text())
concepts = list(concepts_raw.keys()) if isinstance(concepts_raw, dict) else [str(c) for c in concepts_raw]
concept_emb = embed_concepts(model, concepts)

loc_mc = center_renorm(loc_emb, center=True)
con_mc = center_renorm(concept_emb, center=True)
cont_labels, _ = get_continents(df)
print(f"loc_mc: {loc_mc.shape}  |  con_mc: {con_mc.shape}  |  concepts: {len(concepts)}")

Concept emb: 100%|██████████| 1/1 [00:00<00:00,  3.27it/s]

loc_mc: torch.Size([8762, 256])  |  con_mc: torch.Size([1021, 256])  |  concepts: 1021


In [61]:
concepts

['mix',
 'landscape',
 'house',
 'building',
 'patch',
 'tree',
 'vegetation',
 'road',
 'area',
 'street',
 'section',
 'center',
 'water',
 'row',
 'mixture',
 'patchwork',
 'greenery',
 'field',
 'land',
 'body',
 'river',
 'terrain',
 'network',
 'valley',
 'combination',
 'right',
 'scene',
 'structure',
 'vehicle',
 'highway',
 'shade',
 'crop',
 'cluster',
 'shrub',
 'variety',
 'clearing',
 'series',
 'image',
 'ridge',
 'garden',
 'arid',
 'direction',
 'side',
 'stage',
 'forest',
 'car',
 'canal',
 'pathway',
 'path',
 'driveway',
 'intersection',
 'size',
 'snow',
 'left',
 'yard',
 'type',
 'pattern',
 'growth',
 'grid',
 'shape',
 'color',
 'stream',
 'variation',
 'infrastructure',
 'portion',
 'warehouse',
 'cultivation',
 'grass',
 'roadway',
 'soil',
 'traffic',
 'backyard',
 'region',
 'view',
 'farmland',
 'parallel',
 'edge',
 'lawn',
 'waterway',
 'boundary',
 'green',
 'stretch',
 'pond',
 'lane',
 'trail',
 'grassland',
 'park',
 'texture',
 'ice',
 'roundabout'

## 4 · Dimensionality reduction

In [120]:
combined = torch.cat([loc_mc, con_mc]).numpy()
n = len(loc_mc)
Y_umap = umap_reduce(combined, n_components=2, n_neighbors=200, min_dist=0.8, metric="cosine", pca_dim=50)
Y_umap_loc, Y_umap_con = Y_umap[:n], Y_umap[n:]
print("Reductions done.")

Reductions done.


## 5 · Plot

In [121]:
RC_PARAMS = {
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "mathtext.fontset":   "stix",
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
    "axes.titlesize":     14,
    "axes.labelsize":     12,
    "font.size":          12,
    "legend.fontsize":    11,
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "axes.linewidth":     0.8,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.04,
    "figure.facecolor":   "white",
    "axes.facecolor":     "white",
}

In [ ]:
CONTINENT_PALETTE = {
    "Africa":        "#E8761B",   # orange
    "Asia":          "#D62728",   # red
    "Europe":        "#2CA02C",   # green
    "North America": "#1F77B4",   # blue
    "South America": "#9467BD",   # purple
    "Oceania":       "#17BECF",   # cyan/teal
    "Antarctica":    "#BCBD22",   # yellow-green
    "Unknown":       "#999999",   # mid-gray (readable on white)
}
_KNOWN_CONTINENTS = frozenset(CONTINENT_PALETTE) - {"Unknown"}
CONCEPT_COLOR = "#E91C8E"   # vivid magenta — distinct from all continent colours

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

def label_overlapping_concepts(
    ax,
    Y_loc,
    Y_con,
    names=None,
    radius=0.25,
    min_neighbors=20,
    jitter=0.4,
):
    nn = NearestNeighbors(radius=radius).fit(Y_loc)
    neighbor_counts = np.array([
        len(nn.radius_neighbors([p], return_distance=False)[0])
        for p in Y_con
    ])

    for i, cnt in enumerate(neighbor_counts):
        if cnt < min_neighbors:
            continue

        x, y = Y_con[i]

        dx = ((np.sin(i * 12.9898) * 43758.5453) % 1 - 0.5) * jitter
        dy = ((np.sin(i * 78.233)  * 12345.6789) % 1 - 0.5) * jitter

        label = names[i] if names is not None else str(i)

        ax.text(
            x + dx - 0.2,
            y + dy + 0.5,
            label,
            fontsize=8,
            ha="center",
            va="bottom",
            zorder=20,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85),
        )

In [142]:
def _draw_panel(ax, Y_loc, Y_con, cont_labels, title=None,
                loc_size=9, con_size=75, loc_alpha=0.6):
    ca = np.array(cont_labels)
    order = ["Unknown"] + [c for c in CONTINENT_PALETTE if c != "Unknown" and c in set(ca)]
    for cont in order:
        m = ca == cont
        if not m.any():
            continue
        print(cont)
        ax.scatter(Y_loc[m, 0], Y_loc[m, 1], s=loc_size, marker="o", linewidths=0,
                   c=[CONTINENT_PALETTE[cont]], alpha=loc_alpha, rasterized=True)

    # shadow (draw slightly offset, darker, bigger)
    ax.scatter(
        Y_con[:, 0] + 0.05, Y_con[:, 1] - 0.05,
        s=con_size * 1.15,
        marker="^",
        color="black",
        alpha=0.5,
        linewidths=0,
        zorder=4,
        rasterized=True
    )
    ax.scatter(
        Y_con[:, 0], Y_con[:, 1],
        s=con_size,
        marker="^",
        zorder=6,
        facecolors=CONCEPT_COLOR,
        edgecolors="white",   # <-- important for separation
        linewidths=0.1,
        rasterized=True
    )

    if title is not None:
        ax.set_title(title, pad=8, fontweight="regular")
    ax.set_xticks([]); ax.set_yticks([])
    ax.tick_params(left=False, bottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect("equal", adjustable="box")

    all_pts = np.concatenate([Y_loc, Y_con])
    for dim, setter in enumerate([ax.set_xlim, ax.set_ylim]):
        lo, hi = all_pts[:, dim].min(), all_pts[:, dim].max()
        pad = 0.04 * (hi - lo)
        setter(lo - pad, hi + pad)
    label_overlapping_concepts(ax, Y_loc, Y_con, names=concepts)

In [ ]:
def _mk_handle(label, marker, color, size=7, edgecolor="None", edgewidth=0):
    return Line2D([0], [0], marker=marker, linestyle="None", markersize=size,
                  markerfacecolor=color, markeredgecolor=edgecolor,
                  markeredgewidth=edgewidth, label=label)


def _add_legend(ax, fig, present, bottom=0.22):
    # ── Point-type legend (inside plot, upper right) ──────────────────────────
    type_handles = [
        _mk_handle("Location", "o", "#888", size=6),
        _mk_handle("Concept",  "^", CONCEPT_COLOR, size=8, edgecolor="white", edgewidth=0.5),
    ]
    leg_types = ax.legend(
        handles=type_handles,
        title="Point type",
        title_fontsize=8,
        loc="upper right",
        frameon=True,
        framealpha=0.9,
        edgecolor="#cccccc",
        fontsize=9,
        handletextpad=0.5,
        borderpad=0.6,
        alignment="center",
    )
    ax.add_artist(leg_types)

    # ── Continent legend (below the figure) ───────────────────────────────────
    cont_handles = [_mk_handle(c, "o", CONTINENT_PALETTE[c], size=7) for c in present]
    ncol = min(len(cont_handles), 4)
    fig.legend(
        handles=cont_handles,
        title="Region",
        title_fontsize=8,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.0),
        ncol=ncol,
        frameon=False,
        fontsize=9,
        columnspacing=1.2,
        handletextpad=0.4,
        alignment="center",
    )
    fig.subplots_adjust(bottom=bottom)

In [ ]:
for Y_loc, Y_con, method, stem in [
    (Y_umap_loc, Y_umap_con, "UMAP", "umap_mean_centered"),
]:
    matplotlib.rcParams.update(RC_PARAMS)
    fig, ax = plt.subplots(figsize=(8, 7))
    _draw_panel(ax, Y_loc, Y_con, cont_labels, title=None)
    present = [c for c in CONTINENT_PALETTE if c in set(cont_labels)]
    _add_legend(ax, fig, present)
    plt.show()